In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import shapiro, levene, mannwhitneyu

#importing the cleaned dataset
df = pd.read_csv("data_cleaned.csv")

# parse the date column to datetime format
df['date'] = pd.to_datetime(df['date'])


In [ ]:
# This code separates daily study hours into two groups—menstrual and non-menstrual—
# and prints the sample size for each group. These sample sizes are essential for
# evaluating statistical test assumptions and interpreting the analysis.

period_hours = df[df['is_period'] == 1]['study_hours_daily']
nonperiod_hours = df[df['is_period'] == 0]['study_hours_daily']

print("Period sample size:", len(period_hours))
print("Non-period sample size:", len(nonperiod_hours))


Period sample size: 43
Non-period sample size: 136


In [ ]:
# Shapiro–Wilk test evaluates whether each group follows a normal distribution.
# If p < 0.05, the distribution significantly deviates from normality.
print("Shapiro-Wilk Normality Test Results\n")

# Normality test for the period group
stat_p, p_p = shapiro(period_hours)

# Normality test for the non-period group
stat_np, p_np = shapiro(nonperiod_hours)

print(f"Period group → W={stat_p:.4f}, p-value={p_p:.5f}")
print(f"Non-period group → W={stat_np:.4f}, p-value={p_np:.5f}")

if p_p < 0.05:
    print("\nPeriod group is NOT normally distributed.")
else:
    print("\nPeriod group is normally distributed.")

if p_np < 0.05:
    print("Non-period group is NOT normally distributed.")
else:
    print("Non-period group is normally distributed.")


Shapiro-Wilk Normality Test Results

Period group → W=0.8387, p-value=0.00003
Non-period group → W=0.8627, p-value=0.00000

Period group is NOT normally distributed.
Non-period group is NOT normally distributed.


In [ ]:
# Levene’s test checks whether the variances of two groups are statistically equal.
stat_l, p_l = levene(period_hours, nonperiod_hours)

print("Levene Variance Equality Test")
print(f"Statistic={stat_l:.4f}, p-value={p_l:.5f}")

if p_l < 0.05:
    print("\nVariances are NOT equal.")
else:
    print("\nVariances are equal.")


Levene Variance Equality Test
Statistic=0.7669, p-value=0.38237

Variances are equal.


In [ ]:
#Since both the normality and variance assumptions are violated,a parametric T-test is NOT appropriate.
#The correct statistical test is: Mann–Whitney U Test (non-parametric)
# Mann–Whitney U test compares the distributions of two independent groups.
# alternative="less" tests the hypothesis that: period_hours < nonperiod_hours

u_stat, p_val = mannwhitneyu(period_hours, nonperiod_hours, alternative='less')

print("Mann–Whitney U Test (Period < Non-period?)")
print(f"U statistic = {u_stat:.4f}")
print(f"p-value = {p_val:.6f}")

if p_val < 0.05:
    print("\nRESULT: Significant difference found!")
    print("We REJECT the null hypothesis (H₀).")
    print("Period days significantly REDUCE study hours.")
else:
    print("\nRESULT: No significant difference found.")
    print("We FAIL TO REJECT the null hypothesis.")
    print("Period days do NOT significantly reduce study hours.")


Mann–Whitney U Test (Period < Non-period?)
U statistic = 2888.0000
p-value = 0.450992

RESULT: No significant difference found.
We FAIL TO REJECT the null hypothesis.
Period days do NOT significantly reduce study hours.


In [ ]:
# Effect size for Mann–Whitney U (Rank-Biserial Correlation)
# Interpretation:
# 0.1 = small effect, 0.3 = medium effect, 0.5 = large effect

n1 = len(period_hours)
n2 = len(nonperiod_hours)

rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
print("Effect Size (Rank-Biserial Correlation):", round(rank_biserial, 4))


Effect Size (Rank-Biserial Correlation): 0.0123
